# 1. PLM (Pre-trained Language Model)
- PLM(Pre-trained Language Model)은 대량의 텍스트 데이터를 사전 학습하여 자연어 이해와 생성 능력을 갖춘 인공지능 모델입니다. 
- 대표적으로 BERT, GPT, T5 등이 있으며, 이들은 대규모 데이터에서 단어의 의미와 문맥을 학습한 후, 특정 작업(예: 문장 분류, 번역, 질의응답 등)에 맞게 추가 학습(Fine-tuning)하여 활용됩니다. 
- PLM은 문맥을 고려한 자연어 처리 능력이 뛰어나며, 다양한 언어 기반 AI 애플리케이션에서 핵심 기술로 사용됩니다.

In [1]:
import pandas as pd
import numpy as np
import random
import time
import datetime
import csv
import os
import torch
import torch.nn.functional as F

from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from datasets import load_dataset
from torch.optim import AdamW
from transformers import pipeline
from transformers import BertTokenizer
from transformers import BertForSequenceClassification, BertConfig
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, hamming_loss

c:\DEV\AIML\4_MULTIMODAL\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. K-MHaS 데이터셋

In [2]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path.cwd()
REPO_DIR = PROJECT_DIR / 'K-MHaS'
DATA_DIR = REPO_DIR / 'data'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/adlnlp/K-MHaS.git', str(REPO_DIR)], check=True)
else:
    print(f'이미 저장소가 존재합니다: {REPO_DIR}')

print('K-MHaS 폴더:')
for p in REPO_DIR.iterdir():
    print(' -', p.name)

print('\ndata 폴더:')
for p in DATA_DIR.iterdir():
    print(' -', p.name)

K-MHaS 폴더:
 - .git
 - data
 - README.md
 - resource

data 폴더:
 - kmhas_test.txt
 - kmhas_train.txt
 - kmhas_valid.txt


In [3]:
train_file = DATA_DIR / 'kmhas_train.txt'

with train_file.open('r', encoding='utf-8') as f:
    for _ in range(3):
        print(f.readline().rstrip())

document	label
"자한당틀딱들.. 악플질 고만해라."	2,4
정치적으로 편향된 평론한은 분은 별로...	8


In [ ]:
# split='train'는 실제 train의미라기보다 Hugging Face가 생성한 단일 split를 바로 꺼내겠다는 의미
train = load_dataset('csv', data_files=str(DATA_DIR / 'kmhas_train.txt'), delimiter='\t', split='train')
validation = load_dataset('csv', data_files=str(DATA_DIR / 'kmhas_valid.txt'), delimiter='\t', split='train')
test = load_dataset('csv', data_files=str(DATA_DIR / 'kmhas_test.txt'), delimiter='\t', split='train')

print(train)
print(validation)
print(test)

Generating train split: 78977 examples [00:00, 185554.28 examples/s]
Generating train split: 8776 examples [00:00, 141520.01 examples/s]
Generating train split: 21939 examples [00:00, 180447.84 examples/s]

Dataset({
    features: ['document', 'label'],
    num_rows: 78977
})
Dataset({
    features: ['document', 'label'],
    num_rows: 8776
})
Dataset({
    features: ['document', 'label'],
    num_rows: 21939
})


In [6]:
train[0]

{'document': '자한당틀딱들.. 악플질 고만해라.', 'label': '2,4'}

In [8]:
train[0]['document']

'자한당틀딱들.. 악플질 고만해라.'

In [10]:
train[0]['label']

'2,4'

### 특수 토큰 추가
- [CLS]: 문장 시작에 배치되는 분류용 토큰
- [SEP]: 문장의 끝 또는 문장 사이를 구분하는 토큰

In [12]:
train_sentences = list(map(lambda x: '[CLS] ' + str(x) + ' [SEP]', train['document']))
validation_sentences = list(map(lambda x: '[CLS] ' + str(x) + ' [SEP]', validation['document']))
test_sentences = list(map(lambda x: '[CLS] ' + str(x) + ' [SEP]', test['document']))

In [13]:
train_sentences[:5]

['[CLS] 자한당틀딱들.. 악플질 고만해라. [SEP]',
 '[CLS] 정치적으로 편향된 평론한은 분은 별로... [SEP]',
 '[CLS] 적당히좀 쳐먹지.그랬냐??? 안그래도 문재인 때문에 나라 엉망진창인데... [SEP]',
 '[CLS] 안서는 아재들 풀발기 ㅋㄲㅋ [SEP]',
 '[CLS] 우와 ㅋ 능력자 [SEP]']

In [14]:
# 멀티라벨 있으면 ','로 split하고 숫자로 return
def parse_label(s):
    return [int(x) for x in str(s).split(',')]

In [15]:
train_y = [parse_label(s) for s in train['label']]
validation_y = [parse_label(s) for s in validation['label']]
test_y = [parse_label(s) for s in test['label']]

In [16]:
train_y

[[2, 4],
 [8],
 [2],
 [4],
 [8],
 [8],
 [8],
 [8],
 [3],
 [2, 3],
 [8],
 [8],
 [3],
 [8],
 [8],
 [8],
 [3],
 [0],
 [8],
 [8],
 [3],
 [8],
 [3],
 [8],
 [8],
 [3],
 [3],
 [3],
 [8],
 [1, 5],
 [8],
 [5],
 [0],
 [3],
 [0],
 [8],
 [1, 5],
 [0],
 [3],
 [0],
 [7],
 [8],
 [2],
 [8],
 [8],
 [8],
 [2, 7],
 [8],
 [8],
 [3],
 [3],
 [8],
 [1],
 [8],
 [1],
 [4],
 [3, 4],
 [1, 5],
 [8],
 [5],
 [8],
 [1],
 [2],
 [8],
 [8],
 [3],
 [8],
 [8],
 [8],
 [8],
 [8],
 [5],
 [2],
 [8],
 [5],
 [2],
 [8],
 [3],
 [8],
 [8],
 [8],
 [8],
 [8],
 [8],
 [1, 3],
 [0],
 [4],
 [8],
 [0, 2],
 [8],
 [3],
 [8],
 [2, 3, 4],
 [2, 3],
 [3, 5],
 [3, 5],
 [3],
 [0],
 [0, 3],
 [8],
 [8],
 [2],
 [8],
 [8],
 [8],
 [8],
 [2],
 [5],
 [8],
 [1],
 [2],
 [2, 3],
 [4],
 [3],
 [8],
 [2],
 [8],
 [0],
 [3],
 [3],
 [8],
 [8],
 [8],
 [0],
 [3],
 [0, 5],
 [8],
 [8],
 [8],
 [8],
 [8],
 [8],
 [5],
 [8],
 [8],
 [2, 4],
 [8],
 [3, 5],
 [8],
 [3],
 [8],
 [8],
 [0],
 [8],
 [0, 2],
 [8],
 [8],
 [2],
 [8],
 [8],
 [8],
 [2],
 [0, 5],
 [8],
 [2, 3],
 [0]

- 신경망의 출력층은 클래스마다 하나의 값을 출력해야 함
- 예) [2, 3] 같은 가변 길이 label을 고정 길이 벡터로 바꿔야 함
```
예) 
클래스가 0~4이고 정답이 [1, 3]이라면
[0, 1, 0, 1, 0] # multi-hot encoding
```

In [ ]:
# 학습 데이터에 등장하는 클래스 종류를 파악
# num_labels: 출력 클래스 개수
enc = MultiLabelBinarizer()
enc.fit(train_y)
num_labels = len(enc.classes_)
num_labels, enc.classes_

(9, array([0, 1, 2, 3, 4, 5, 6, 7, 8]))

In [19]:
train_labels = torch.tensor(enc.transform(train_y), dtype=torch.float32)
validation_labels = torch.tensor(enc.transform(validation_y), dtype=torch.float32)
test_labels = torch.tensor(enc.transform(test_y), dtype=torch.float32)

In [20]:
train_labels

tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        [1., 0., 0.,  ..., 0., 0., 0.]])

In [21]:
test_sentences[:5]

['[CLS] 그만큼 길예르모가 잘했다고 보면되겠지 기대되네 셰이프 오브 워터 [SEP]',
 '[CLS] 1. 8넘의 문재앙 [SEP]',
 '[CLS] 문재인 정권의 내로남불은 타의 추종을 불허하네. 자한당 욕할거리도 없음. [SEP]',
 '[CLS] 짱개들 지나간 곳은 폐허된다 ㅋㅋ [SEP]',
 '[CLS] 곱창은 자갈치~~~~~ [SEP]']

In [22]:
test[:5]

{'document': ['그만큼 길예르모가 잘했다고 보면되겠지 기대되네 셰이프 오브 워터',
  '1. 8넘의 문재앙',
  '문재인 정권의 내로남불은 타의 추종을 불허하네. 자한당 욕할거리도 없음.',
  '짱개들 지나간 곳은 폐허된다 ㅋㅋ',
  '곱창은 자갈치~~~~~'],
 'label': ['8', '2,3', '2', '0', '8']}

In [23]:
test_labels[:5]

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 1., 1., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1.]])

- train/validation/test 각각 별도의 MultiLabelBinarizer를 만들지 않고, train에서 fit한 동일 enc 객체를 사용했음 > 클래스 인덱스의 의미가 일치

In [24]:
# KLU:E-BERT이 맞는 vocabulary와 tokenizer 설정을 내려받아 로드
tokenizer = BertTokenizer.from_pretrained('klue/bert-base')

In [25]:
# 문장 길이는 서로 다르지만 mini-batch는 동일한 크기의 Tensor여야 하므로, 짧은 문장에는 [PAD]를 붙이고, 긴 문장은 MAX_LEN=128로 자름
MAX_LEN = 128  # [CLS], [SEP]를 포함한 최대 토큰 수

def data_to_tensor(sentences, labels, tokenizer, max_len=MAX_LEN):
    # tokenizer에서 사용하는 [PAD] 토큰의 ID
    pad_id = tokenizer.pad_token_id

    # 문장별 토큰 ID Tensor를 저장할 리스트
    input_ids = []

    for sent in sentences:
        # None은 빈 문자열로, 나머지는 문자열로 변환
        sent = "" if sent is None else str(sent)

        # 문장 시작과 끝에 특수 토큰 추가
        # 이미 붙어 있으면 중복해서 추가하지 않음
        if not sent.startswith("[CLS]"):
            sent = "[CLS] " + sent

        if not sent.endswith("[SEP]"):
            sent = sent + " [SEP]"

        # 문장을 tokenizer 기준으로 토큰으로 분리
        # 한 단어가 여러 토큰으로 나뉠 수도 있음
        tokens = tokenizer.tokenize(sent)

        # 최대 토큰 수를 넘으면 뒤쪽을 자름
        if len(tokens) > max_len:
            tokens = tokens[:max_len]

            # 잘리면서 [SEP]가 사라졌다면 마지막 토큰을 [SEP]로 교체
            if tokens[-1] != "[SEP]":
                tokens[-1] = "[SEP]"

        # 토큰 문자열을 사전에 등록된 정수 ID로 변환
        ids = tokenizer.convert_tokens_to_ids(tokens)

        # Embedding 입력에 사용할 정수형 Tensor로 변환
        input_ids.append(torch.tensor(ids, dtype=torch.long))

    # 현재 문장들 중 가장 긴 길이에 맞춰 나머지 문장 뒤에 PAD 추가
    # batch_first=True → shape: (문장 수, 토큰 수)
    padded_inputs = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=pad_id
    )

    # 가장 긴 문장도 max_len보다 짧으면
    # 오른쪽에 PAD를 더 붙여 정확히 max_len으로 맞춤
    if padded_inputs.size(1) < max_len:
        padded_inputs = F.pad(
            padded_inputs,
            (0, max_len - padded_inputs.size(1)),
            value=pad_id
        )
    else:
        # 모든 문장에서 앞쪽 max_len개 토큰만 유지
        padded_inputs = padded_inputs[:, :max_len]

    # 실제 토큰은 1.0, PAD는 0.0
    # 모델이 PAD를 참고하지 않도록 전달할 마스크
    attention_masks = (padded_inputs != pad_id).float()

    # 다중 라벨 분류에 사용할 정답을 float32 Tensor로 변환
    if torch.is_tensor(labels):
        tensor_labels = labels.to(dtype=torch.float32)
    else:
        tensor_labels = torch.tensor(labels, dtype=torch.float32)

    return padded_inputs, tensor_labels, attention_masks

In [27]:
# 학습·검증·테스트 문장과 정답 라벨을 Tensor로 변환하고, 문장 길이를 128토큰으로 맞춘 입력 ID와 PAD를 구별하는 Attention Mask를 생성

# *_inputs: token ID Tensor
# *_labels: multi-hot 정답 Tensor
# *_mask: attention mask Tensor
train_inputs, train_labels, train_mask = data_to_tensor(train_sentences, train_labels, tokenizer)
validation_inputs, validation_labels, validation_mask = data_to_tensor(validation_sentences, validation_labels, tokenizer)
test_inputs, test_labels, test_mask = data_to_tensor(test_sentences, test_labels, tokenizer)

In [28]:
train_inputs, train_labels, train_mask

(tensor([[    2,  1517,  2470,  ...,     0,     0,     0],
         [    2,  3713, 11187,  ...,     0,     0,     0],
         [    2, 11466,  2505,  ...,     0,     0,     0],
         ...,
         [    2, 31035,  2425,  ...,     0,     0,     0],
         [    2,     1, 10758,  ...,     0,     0,     0],
         [    2,  1439,  2075,  ...,     0,     0,     0]]),
 tensor([[0., 0., 1.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 1.],
         [0., 0., 1.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 1.],
         [0., 0., 0.,  ..., 0., 0., 1.],
         [1., 0., 0.,  ..., 0., 0., 0.]]),
 tensor([[1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         ...,
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.]]))

In [29]:
# 학습·검증·테스트의 입력 ID, Attention Mask, 정답을 하나의 데이터셋으로 묶고, 무작위 순서로 최대 32개씩 꺼내는 DataLoader를 생성

batch_size = 32

train_data = TensorDataset(train_inputs, train_mask, train_labels)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

validation_data = TensorDataset(validation_inputs, validation_mask, validation_labels)
validation_sampler = RandomSampler(validation_data)
validation_dataloader = DataLoader(validation_data, sampler=validation_sampler, batch_size=batch_size)

test_data = TensorDataset(test_inputs, test_mask, test_labels)
test_sampler = RandomSampler(test_data)
test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=batch_size)

len(train_labels), len(validation_labels), len(test_labels)

(78977, 8776, 21939)

In [30]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

cuda


# 3. KLUE-BERT 다중 레이블 분류 모델

In [31]:
# 모델 생성
num_labels = len(enc.classes_)
model = BertForSequenceClassification.from_pretrained(
    'klue/bert-base',
    num_labels=num_labels,
    problem_type='multi_label_classification'
)
model.to(DEVICE)
model

c:\DEV\AIML\4_MULTIMODAL\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kesgo\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\DEV\AIML\4_MULTIMODAL\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by d

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [32]:
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
epochs = 10
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [33]:
# time.time()의 차이는 초 단위 숫자 > 시:분:초 형태로 바꾸기 위한 함수
def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))

In [34]:
def multi_label_metrics(predictions, labels, threshold=0.5):
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= threshold)] = 1
    y_true = labels

    accuracy = accuracy_score(y_true, y_pred)
    f1_macro_average = f1_score(y_true=y_true, y_pred=y_pred, average='macro', zero_division=0)
    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro', zero_division=0)
    f1_weighted_average = f1_score(y_true=y_true, y_pred=y_pred, average='weighted', zero_division=0)
    roc_auc = roc_auc_score(y_true, y_pred, average='micro')

    metrics = {
        'accuracy': accuracy,
        'f1_macro_average': f1_macro_average,
        'f1_micro_average': f1_micro_average,
        'f1_weighted_average': f1_weighted_average,
        'roc_auc': roc_auc
    }
    return metrics

In [35]:
seed_val = 2026
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed_val)

In [36]:
# 이전에 계산된 기울기를 초기화
model.zero_grad()

# 전체 학습 데이터를 epochs번 반복 학습
for epoch_i in range(0, epochs):
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))

    t0 = time.time()  # 현재 epoch의 시작 시간
    total_loss = 0   # 현재 epoch의 배치별 loss 누적값

    # 학습 모드로 전환: Dropout 등 학습용 동작 활성화
    model.train()

    # DataLoader에서 미니배치를 하나씩 꺼내 학습
    for step, batch in tqdm(enumerate(train_dataloader)):

        # 500개 배치마다 진행 상황과 경과 시간 출력
        if step % 500 == 0 and step != 0:
            elapsed = format_time(time.time() - t0)
            print(
                '  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(
                    step, len(train_dataloader), elapsed
                )
            )

        # 배치의 모든 Tensor를 모델과 같은 장치(GPU 또는 CPU)로 이동
        batch = tuple(t.to(DEVICE) for t in batch)

        # TensorDataset에 넣었던 순서대로 분리
        b_input_ids, b_input_mask, b_labels = batch

        # 순전파: 입력으로 예측을 계산하고, 정답과 비교해 loss 계산
        # token_type_ids=None: 문장 구분 ID를 별도로 전달하지 않음
        # attention_mask: PAD 위치를 참고하지 않도록 지정
        # labels: loss 계산에 사용할 실제 정답
        outputs = model(
            b_input_ids,
            token_type_ids=None,
            attention_mask=b_input_mask,
            labels=b_labels
        )

        # 이 모델의 반환 형식에서 첫 번째 값인 loss 추출
        loss = outputs[0]

        # loss를 Python 숫자로 꺼내 누적: 평균 학습 loss 출력에 사용
        total_loss += loss.item()

        # 역전파: loss를 줄이기 위한 각 파라미터의 기울기 계산
        loss.backward()

        # 전체 기울기의 L2 norm이 1.0을 넘으면 비율을 유지하며 축소
        # 기울기 폭주를 완화하는 방법이며, BERT에서 반드시 필요한 것은 아님
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # 계산된 기울기로 모델 파라미터 업데이트
        optimizer.step()

        # 설정한 스케줄에 따라 다음 업데이트에 사용할 학습률 조정
        scheduler.step()

        # 다음 배치에 현재 기울기가 누적되지 않도록 초기화
        model.zero_grad()

    # 현재 epoch의 배치별 loss를 단순 평균
    avg_train_loss = total_loss / len(train_dataloader)

    print("")
    print("  Average training loss: {0:.4f}".format(avg_train_loss))
    print(
        "  Training epoch took: {:}".format(
            format_time(time.time() - t0)
        )
    )

======== Epoch 1 / 10 ========


500it [03:17,  2.73it/s]

  Batch   500  of  2,469.    Elapsed: 0:03:17.


1000it [06:16,  2.81it/s]

  Batch 1,000  of  2,469.    Elapsed: 0:06:16.


1500it [09:17,  2.79it/s]

... (91 outputs left out)

2000it [11:19,  3.01it/s]

  Batch 2,000  of  2,469.    Elapsed: 0:11:19.


2469it [13:57,  2.95it/s]


  Average training loss: 0.0107
  Training epoch took: 0:13:58


In [38]:
t0 = time.time()
model.eval()
accum_logits, accum_label_ids = [], []

for batch in validation_dataloader:
    batch = tuple(t.to(DEVICE) for t in batch)
    b_input_ids, b_input_mask, b_labels = batch

    with torch.no_grad():
        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask)

    logits = outputs[0]
    logits = logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    for b in logits:
        accum_logits.append(list(b))

    for b in label_ids:
        accum_label_ids.append(list(b))

accum_logits = np.array(accum_logits)
accum_label_ids = np.array(accum_label_ids)
results = multi_label_metrics(accum_logits, accum_label_ids)

print("Accuracy: {0:.4f}".format(results['accuracy']))
print("F1 (Macro) Score: {0:.4f}".format(results['f1_macro_average']))
print("F1 (Micro) Score: {0:.4f}".format(results['f1_micro_average']))
print("F1 (Weighted) Score: {0:.4f}".format(results['f1_weighted_average']))
print("ROC-AUC: {0:.4f}".format(results['roc_auc']))

Accuracy: 0.8038
F1 (Macro) Score: 0.7584
F1 (Micro) Score: 0.8494
F1 (Weighted) Score: 0.8489
ROC-AUC: 0.9129


In [37]:
from pathlib import Path

MODEL_DIR = Path.cwd() / 'model'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('모델 저장 폴더:', MODEL_DIR.resolve())

모델 저장 폴더: C:\DEV\AIML\4_MULTIMODAL\model


In [39]:
model_path = MODEL_DIR / 'BERT_multilabel_model.pt'
torch.save(model.state_dict(), model_path)
print('저장 완료:', model_path.resolve())

저장 완료: C:\DEV\AIML\4_MULTIMODAL\model\BERT_multilabel_model.pt


In [40]:
state_dict = torch.load(model_path, map_location=DEVICE)
model.load_state_dict(state_dict)
model.to(DEVICE)
print('모델 로드 완료')

모델 로드 완료


In [41]:
t0 = time.time()
model.eval()
accum_logits, accum_label_ids = [], []

for step, batch in tqdm(enumerate(test_dataloader)):
    if step % 100 == 0 and not step == 0:
        elapsed = format_time(time.time() - t0)
        print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(test_dataloader), elapsed))

    batch = tuple(t.to(DEVICE) for t in batch)
    b_input_ids, b_input_mask, b_labels = batch

    with torch.no_grad():
        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask)

    logits = outputs[0]
    logits = logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    for b in logits:
        accum_logits.append(list(b))

    for b in label_ids:
        accum_label_ids.append(list(b))

accum_logits = np.array(accum_logits)
accum_label_ids = np.array(accum_label_ids)
results = multi_label_metrics(accum_logits, accum_label_ids)

print("Accuracy: {0:.4f}".format(results['accuracy']))
print("F1 (Macro) Score: {0:.4f}".format(results['f1_macro_average']))
print("F1 (Micro) Score: {0:.4f}".format(results['f1_micro_average']))
print("F1 (Weighted) Score: {0:.4f}".format(results['f1_weighted_average']))
print("ROC-AUC: {0:.4f}".format(results['roc_auc']))

101it [00:12,  8.56it/s]

  Batch   100  of    686.    Elapsed: 0:00:12.


201it [00:23,  8.54it/s]

  Batch   200  of    686.    Elapsed: 0:00:24.


301it [00:35,  8.46it/s]

  Batch   300  of    686.    Elapsed: 0:00:35.


... (4 outputs left out)

601it [01:10,  8.28it/s]

  Batch   600  of    686.    Elapsed: 0:01:11.


686it [01:20,  8.50it/s]


Accuracy: 0.8063
F1 (Macro) Score: 0.7624
F1 (Micro) Score: 0.8514
F1 (Weighted) Score: 0.8511
ROC-AUC: 0.9148


# 4. Hugging Face Pipeline으로 추론 인터페이스 만들기
- Pipeline()은 tokenizer > Tensor 변환 > model 추론 > 결과 후처리를 하나로 묶어주는 Hugging Face의 API

In [42]:
pipe_device = 0 if torch.cuda.is_available() else -1

# GPU가 있으면 device: 0, CPU 사용 device: -1
# truncation=True: 너무 긴 문장을 자름 (max_length 기준)
# function_to_apply='sigmoid': logits를 각 클래스의 0~1 score로 변환
# top_k=None: 특정 상위 몇 개만 자르지 말고 모든 label 결과를 반환
pipe = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=pipe_device,
    truncation=True,
    max_length=512,
    function_to_apply='sigmoid',
    top_k=None
)

In [43]:
result = pipe("잼민이들 왜 그렇게 민폐냐?")
print(result)

[[{'label': 'LABEL_8', 'score': 0.9992172718048096}, {'label': 'LABEL_3', 'score': 0.00035893107997253537}, {'label': 'LABEL_4', 'score': 0.00026049595908261836}, {'label': 'LABEL_2', 'score': 0.0002297898317920044}, {'label': 'LABEL_5', 'score': 0.00022448098752647638}, {'label': 'LABEL_0', 'score': 0.00018482438463252038}, {'label': 'LABEL_6', 'score': 0.00014535232912749052}, {'label': 'LABEL_1', 'score': 0.00011809237184934318}, {'label': 'LABEL_7', 'score': 6.427631160477176e-05}]]


In [44]:
label_dict = {'LABEL_0' : '출신차별', 'LABEL_1' : '외모차별', 'LABEL_2' : '정치성향차별', \
              'LABEL_3': '혐오욕설', 'LABEL_4': '연령차별', 'LABEL_5': '성차별', 'LABEL_6' : '인종차별', \
              'LABEL_7': '종교차별', 'LABEL_8': '해당사항없음'}

In [45]:
def prediction(text):
    result = pipe(text)
    return [label_dict[res['label']] for res in result[0] if res['score'] > 0.5]

In [47]:
prediction("너희 나라로 돌아가라")

['출신차별']

In [49]:
prediction("못생겼다")

['외모차별']

['출신차별']